# 02 - Feature Engineering 

In [2]:
import pandas as pd
df = pd.read_csv("../data/processed/cleaned_data.csv")
print(df.shape)
print(df.columns.tolist())

(6232, 13)
['s.no', 'helpfulVoteCount', 'productASIN', 'rating', 'reviewID', 'reviewPosition', 'reviewTitle', 'reviewURL', 'verifiedPurchase', 'sentiment_score', 'at', 'content_clean', 'text_length']


### Drop columns with unusable text, using the tf-idf matrix

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

df = df.dropna(subset=["content_clean"])
df = df[df["content_clean"].str.strip() != ""]

tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['content_clean'])
print("Shape of the TF-IDF matrix:", X.shape)


Shape of the TF-IDF matrix: (6232, 5000)


### dropping near duplicates via cosine similarity

In [6]:
from sklearn.neighbors import NearestNeighbors

n_neighbors = min(2, X.shape[0])
nn = NearestNeighbors(n_neighbors=n_neighbors, metric='cosine').fit(X)
distances, indices = nn.kneighbors(X)
df['near_dup_score'] = 1 - distances[:, -1]
df['is_near_dup'] = (df['near_dup_score'] > 0.9).astype(int)

### Burst detection (per product)

In [7]:
df["at"] = pd.to_datetime(df["at"], errors="coerce")

# Per-product daily counts
daily_counts = df.groupby(['productASIN', df['at'].dt.date]).size()
mean_c, std_c = daily_counts.mean(), daily_counts.std()
burst_keys = set(daily_counts[daily_counts > mean_c + 2*std_c].index)

df['product_date_key'] = list(zip(df['productASIN'], df['at'].dt.date))
df['is_burst_day'] = df['product_date_key'].isin(burst_keys).astype(int)
df = df.drop(columns=['product_date_key'])

### Rating deviation (we don't include the review we are rating)

In [8]:
def loo_mean(group):
    n = len(group)
    if n <= 1:
        return group * 0
    total = group.sum()
    return (total - group) / (n - 1)

product_loo_mean = df.groupby('productASIN')['rating'].transform(loo_mean)
df['rating_deviation'] = (df['rating'] - product_loo_mean).abs()

### Verified purchase signal

In [9]:
df['is_unverified'] = (df['verifiedPurchase'] == False).astype(int)

### Linguistic features

In [10]:
import numpy as np

df['word_count'] = df['content_clean'].str.split().apply(len)

df['avg_word_len'] = df['content_clean'].apply(
    lambda t: np.mean([len(w) for w in t.split()]) if t.split() else 0
)

df['positive_superlative_count'] = df['content_clean'].str.count(
    r'\b(best|amazing|perfect|excellent|incredible)\b'
)
df['negative_superlative_count'] = df['content_clean'].str.count(
    r'\b(worst|terrible|awful|horrible)\b'
)

df['exclamation_count'] = df['content_clean'].str.count('!')

### Scaling features

In [11]:
from sklearn.preprocessing import MinMaxScaler

feature_cols = ['near_dup_score', 'is_burst_day', 'is_unverified', 'rating_deviation',
                'exclamation_count', 'word_count', 'avg_word_len',
                'positive_superlative_count', 'negative_superlative_count']

scaler = MinMaxScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

print(df[feature_cols].describe())

       near_dup_score  is_burst_day  is_unverified  rating_deviation  \
count     6232.000000    6232.00000    6232.000000       6232.000000   
mean         0.411824       0.09708       0.024551          0.129347   
std          0.140194       0.29609       0.154764          0.133189   
min          0.000000       0.00000       0.000000          0.000000   
25%          0.322546       0.00000       0.000000          0.044444   
50%          0.374286       0.00000       0.000000          0.088889   
75%          0.457036       0.00000       0.000000          0.155556   
max          1.000000       1.00000       1.000000          1.000000   

       exclamation_count   word_count  avg_word_len  \
count        6232.000000  6232.000000   6232.000000   
mean            0.010769     0.022835      0.338578   
std             0.028867     0.033568      0.065542   
min             0.000000     0.000000      0.000000   
25%             0.000000     0.005656      0.299325   
50%             0.000

### Saving

In [12]:
feature_cols_to_save = ['reviewID', 'productASIN', 'content_clean', 'rating'] + feature_cols
df[feature_cols_to_save].to_csv('../data/processed/featured_reviews.csv', index=False)
print('Saved:', df[feature_cols_to_save].shape)

Saved: (6232, 13)
